In [ ]:
import gc
import json
import os
import subprocess
import sys
from pathlib import Path

REPO_REV = '81b4d39'
REPO_ROOT = Path('/kaggle/working/spider')
subprocess.run(['git', 'clone', 'https://github.com/yogesh-dhande/spider.git', str(REPO_ROOT)], check=True)
subprocess.run(['git', '-C', str(REPO_ROOT), 'checkout', REPO_REV], check=True)
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT / 'src'))
os.environ['HF_HUB_DOWNLOAD_TIMEOUT'] = '300'
os.environ['HF_HUB_ETAG_TIMEOUT'] = '60'
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'


In [ ]:
%pip install -q --progress-bar off -r requirements/experiment2-kaggle.txt


In [ ]:
from spider.exp4_data import find_exp2_initial_adapter, find_exp4_data
from spider.workflow import gpu_summary

prepared = find_exp4_data('/kaggle/input')
initial_adapter = find_exp2_initial_adapter('/kaggle/input')
os.environ['SPIDER_DATA_DIR'] = str(prepared)
print({'event': 'baseline_inputs', 'prepared': str(prepared), 'adapter': str(initial_adapter), **gpu_summary()}, flush=True)


In [ ]:
from spider.action_evaluate import evaluate_actions

_, base_metrics = evaluate_actions('configs/experiment4.yaml', 'action-base-shard-00-of-02', None, split='development', shard_index=0, num_shards=2)
print({'event': 'base_action_shard_complete', 'metrics': base_metrics}, flush=True)


In [ ]:
import torch

gc.collect()
torch.cuda.empty_cache()
_, exp2_metrics = evaluate_actions('configs/experiment4.yaml', 'action-exp002-shard-00-of-02', str(initial_adapter), split='development', shard_index=0, num_shards=2)
print({'event': 'exp002_action_shard_complete', 'metrics': exp2_metrics}, flush=True)
